In [ ]:
!pip install torch torchvision matplotlib pillow opencv-python

import torch
import torchvision.transforms as T
import torchvision.models as models
import torchvision.models.detection as detection
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2


image_path = "/content/sample_data/street.jpg"
image = Image.open(image_path).convert("RGB")

plt.imshow(image)
plt.title("Original Image")
plt.axis("off")
plt.show()

# PART 1: SEMANTIC SEGMENTATION (DeepLabV3)

print("Running Semantic Segmentation...")

semantic_model = models.segmentation.deeplabv3_resnet101(pretrained=True)
semantic_model.eval()

transform = T.Compose([
    T.Resize(520),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
])

input_tensor = transform(image).unsqueeze(0)

with torch.no_grad():
    semantic_output = semantic_model(input_tensor)['out'][0]

semantic_mask = semantic_output.argmax(0).byte().cpu().numpy()

plt.figure(figsize=(8,6))
plt.imshow(semantic_mask)
plt.title("Semantic Segmentation Output")
plt.axis("off")
plt.show()


# PART 2: INSTANCE SEGMENTATION (Mask R-CNN)

print("Running Instance Segmentation...")

instance_model = detection.maskrcnn_resnet50_fpn(pretrained=True)
instance_model.eval()

transform = T.ToTensor()
img_tensor = transform(image)

with torch.no_grad():
    predictions = instance_model([img_tensor])


# Convert to OpenCV format
image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

# Apply masks
for i in range(len(predictions[0]['masks'])):

    score = predictions[0]['scores'][i]

    if score > 0.5:

        mask = predictions[0]['masks'][i,0].mul(255).byte().cpu().numpy()

        image_cv[mask > 127] = [0,255,0]


plt.figure(figsize=(8,6))
plt.imshow(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
plt.title("Instance Segmentation Output")
plt.axis("off")
plt.show()